In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/Sheep_Project/Sheep_Teeth_Dataset"
print(os.listdir(DATASET_DIR))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import class_weight


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


In [ ]:
labels = train_generator.classes
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights_dict = dict(enumerate(weights))
print(class_weights_dict)


In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)


In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("/content/drive/MyDrive/Sheep_Project/accuracy_plot.png")
plt.show()


plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig("/content/drive/MyDrive/Sheep_Project/loss_plot.png")
plt.show()


In [ ]:
def predict_sheep_age(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)[0]

    return preds



In [ ]:
model.save("sheep_age_model.h5")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("sheep_age_detection.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully!")


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image


interpreter = tf.lite.Interpreter(model_path="sheep_age_detection.tflite")
interpreter.allocate_tensors()


input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]['shape'])
print("Output shape:", output_details[0]['shape'])


In [ ]:
import os
import numpy as np
import tensorflow as tf

def test_random_image_from_class(class_folder):

    valid_ext = (".jpg", ".jpeg", ".png")

    images = [f for f in os.listdir(class_folder) if f.lower().endswith(valid_ext)]

    if len(images) == 0:
        print("No valid images found in:", class_folder)
        return

    img_path = os.path.join(class_folder, images[0])
    print("Testing image:", img_path)

    img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0).astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], img_array)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])[0]

    print("Prediction Probabilities:")
    for i, prob in enumerate(output):
        print(f"Class {i}: {prob:.4f}")

    print("Predicted Class:", np.argmax(output))


In [ ]:
test_random_image_from_class("/content/drive/MyDrive/Sheep_Project/Sheep_Teeth_Dataset/class_0_3_12_months")

test_random_image_from_class("/content/drive/MyDrive/Sheep_Project/Sheep_Teeth_Dataset/class_1_1_1_5_years")

test_random_image_from_class("/content/drive/MyDrive/Sheep_Project/Sheep_Teeth_Dataset/class_2_1_5_2_years")

test_random_image_from_class("/content/drive/MyDrive/Sheep_Project/Sheep_Teeth_Dataset/class_3_2_3_years")
